In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("visualizations", exist_ok=True)

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))


In [ ]:


df = pd.read_csv("/kaggle/input/datasets/surajjha101/bigbasket-entire-product-list-28k-datapoints/BigBasket Products.csv")

print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
transactions = (
    df.groupby('category')['sub_category']
    .apply(list)
    .tolist()
)

transactions[:5]

In [ ]:
!pip install mlxtend -q

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import warnings

warnings.filterwarnings("ignore")

transactions = (
    df.groupby('category')['sub_category']
    .apply(list)
    .tolist()
)

te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    te_array,
    columns=te.columns_
)

basket.head()

In [ ]:
transactions = (
    df.groupby('brand')['sub_category']
    .apply(lambda x: list(set(x)))
    .tolist()
)

te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    te_array,
    columns=te.columns_
)

basket.shape

In [ ]:
frequent_items = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

frequent_items = frequent_items.sort_values(
    by='support',
    ascending=False
)

frequent_items.head(10)

In [ ]:
rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.2
)

rules = rules.sort_values(
    by='lift',
    ascending=False
)

rules[
    [
        'antecedents',
        'consequents',
        'support',
        'confidence',
        'lift'
    ]
].head(10)

In [ ]:
import matplotlib.pyplot as plt

top_rules = rules.head(10)

plt.figure(figsize=(10,6))

plt.bar(
    range(len(top_rules)),
    top_rules['lift']
)

plt.xticks(
    range(len(top_rules)),
    [
        f"{list(a)[0]} -> {list(c)[0]}"
        for a, c in zip(
            top_rules['antecedents'],
            top_rules['consequents']
        )
    ],
    rotation=75
)

plt.ylabel("Lift")
plt.title("Top Market Basket Associations")

plt.tight_layout()
plt.show()

In [ ]:
recommendations = rules[
    [
        'antecedents',
        'consequents',
        'confidence',
        'lift'
    ]
].copy()

recommendations['antecedents'] = recommendations[
    'antecedents'
].apply(lambda x: ', '.join(list(x)))

recommendations['consequents'] = recommendations[
    'consequents'
].apply(lambda x: ', '.join(list(x)))

recommendations.head(15)

In [ ]:
print("Top Business Insights:\n")

for _, row in recommendations.head(5).iterrows():
    print(
        f"If customer buys [{row['antecedents']}], "
        f"they are likely to buy "
        f"[{row['consequents']}] "
        f"(Lift={row['lift']:.2f}, "
        f"Confidence={row['confidence']:.2f})"
    )

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_data = rules.pivot_table(
    values='lift',
    index='antecedents',
    columns='consequents'
)

plt.figure(figsize=(12,8))

sns.heatmap(
    heatmap_data,
    annot=False,
    cmap='Blues'
)

plt.title("Association Rules Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

top_items = frequent_items.head(20)

plt.figure(figsize=(14,8))

plt.barh(
    [
        ', '.join(list(item))
        for item in top_items['itemsets']
    ],
    top_items['support']
)

plt.xlabel("Support")
plt.ylabel("Itemsets")
plt.title("Top 20 Frequent Itemsets")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

top_rules = rules.head(15)

G = nx.DiGraph()

for _, row in top_rules.iterrows():

    antecedent = ', '.join(list(row['antecedents']))
    consequent = ', '.join(list(row['consequents']))

    G.add_edge(
        antecedent,
        consequent,
        weight=row['lift']
    )

plt.figure(figsize=(14,10))

pos = nx.spring_layout(G, seed=42)

nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=4000,
    font_size=9,
    arrows=True
)

edge_labels = nx.get_edge_attributes(G, 'weight')

nx.draw_networkx_edge_labels(
    G,
    pos,
    edge_labels={
        k: f"{v:.2f}"
        for k, v in edge_labels.items()
    }
)

plt.title("Market Basket Association Network")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(
    rules['support'],
    rules['confidence'],
    s=rules['lift'] * 50
)

plt.xlabel("Support")
plt.ylabel("Confidence")
plt.title("Support vs Confidence")

plt.tight_layout()
plt.show()